# BLISS User API

Bayesian Light Source Separator (BLISS) is a Bayesian method for deblending and cataloging light sources.

## Installation

In [ ]:
!pip install -e /home/zhteoh/770-bulk-predict

In [ ]:
!pip install bliss-deblender

## Tutorial

In [ ]:
from bliss.api import BlissClient

# bliss_client = BlissClient(cwd="/data/scratch/zhteoh/tutorial")
bliss_client = BlissClient(cwd="/tmp/pytest-of-zhteoh/pytest-417")

### Train the model

#### Generate synthetic image data

In [ ]:
bliss_client.generate(
    n_batches=3, 
    batch_size=64,
    max_images_per_file=128
)

##### Pass additional custom configuration parameters

In [ ]:
# Alter default cached_data_path
bliss_client.cached_data_path = "/data/scratch/zhteoh/tutorial/data/cached_dataset_ms0.02"

bliss_client.generate(
    n_batches=3,  # required
    batch_size=64,  # required
    max_images_per_file=128,  # required
    simulator={"survey": {"prior_config": {"mean_sources": 0.02}}},  # optional
    generate={"file_prefix": "dataset"},  # optional
)

In [ ]:
bliss_client.cached_data_path = "/data/scratch/zhteoh/tutorial/data/cached_dataset_ms0.02"

In [ ]:
# Check that the dataset is generated
!ls /data/scratch/zhteoh/tutorial/data/cached_dataset_ms0.02
!du -sh /data/scratch/zhteoh/tutorial/data/cached_dataset_ms0.02
# !cat /data/scratch/zhteoh/tutorial/dataset/hparams.yaml

print("Dataset:", bliss_client.cached_data_path)
dataset_0 = bliss_client.get_dataset_file(filename="dataset_0.pt")
print(" Size:", len(dataset_0))
print(" Shape:", dataset_0[0]["images"].shape)

#### Train the model

##### Without pretrained weights

In [ ]:
bliss_client.train(weight_save_path="tutorial_encoder/0.pt")

##### With pretrained weights

Download our relevant pretrained weights for your sky survey.

In [ ]:
import os
assert os.path.exists("/data/scratch/zhteoh/tutorial/data/pretrained_models")

bliss_client.load_pretrained_weights_for_survey(survey="sdss", filename="sdss_pretrained.pt")

!ls /data/scratch/zhteoh/tutorial/data/pretrained_models

##### Train on cached generated disk dataset

In [ ]:
bliss_client.train_on_cached_data(
    weight_save_path="tutorial_encoder/0.pt",
    train_n_batches=2,
    batch_size=64,
    val_split_file_idxs=[1],
    pretrained_weights_filename=None,
)

### Run the model

#### Using sample SDSS dataset

##### Get predictions for the sample dataset

In [ ]:
est_cat, est_cat_table, pred_tables = bliss_client.predict_sdss(
    weight_save_path="tutorial_encoder/zscore_five_band.pt",
    # predict={"dataset": {"sdss_fields": [{"run": 94, "camcol": 1, "fields": [12]}, {"run": 3900, "camcol": 6, "fields": [296]}]}},
)

In [ ]:
bliss_client.plot_predictions_in_notebook()

In [ ]:
print("Number of entries:", len(est_cat_table))
est_cat_table[:5].show_in_notebook(display_length=5)

##### Inspect probabilistic predictions

BLISS produces probability distributions on the predicted latent variables.

In [ ]:
print("Number of entries (RCF (94, 1, 12)):", len(pred_tables[(94, 1, 12)]))
pred_tables[(94, 1, 12)][:5].show_in_notebook(display_length=5)

In [ ]:
print("Number of entries (RCF (3900, 6, 269)):", len(pred_tables[(3900, 6, 269)]))
pred_tables[(3900, 6, 269)][:5].show_in_notebook(display_length=5)

##### Save predicted catalog to FITS file

In [ ]:
est_cat_table.write("est_cat.fits", format="fits", overwrite=True)

In [ ]:
# Check that catalog is saved as intended
from astropy.table import Table

est_cat_table = Table.read("est_cat.fits", format="fits")
print("Number of entries:", len(est_cat_table))
est_cat_table.show_in_notebook(display_length=5)

##### Evaluate prediction

In [ ]:
import torch

from bliss.metrics import BlissMetrics
from bliss.surveys.sdss import PhotoFullCatalog

sdss_data_path = "/data/scratch/zhteoh/tutorial/data/sdss"
sdss = SloanDigitalSkySurvey()
photo_cat = PhotoFullCatalog.from_file()

est_cat_cuda = est_cat.to(torch.device("cpu"))
photo_cat_cuda = photo_cat.to(torch.device("cpu"))

metrics = BlissMetrics()
results = metrics(est_cat_cuda, photo_cat_cuda)

print(results)

#### Using user-specified SDSS dataset

##### Download online dataset

In [ ]:
from astropy.coordinates import SkyCoord
from astroquery.sdss import SDSS
from pathlib import Path

pos = SkyCoord('0h8m05.63s +14d50m23.3s', frame='icrs') # 1011/3/44
# pos = SkyCoord("1h8m05.73s +13d10m20.3s", frame="icrs") # 4829/5/27
# pos = SkyCoord("1h2m05.83s -2d11m20.3s", frame="icrs") # 2699/4/71
region = SDSS.query_region(pos, radius="5 arcsec")
run, camcol, field = region["run"][0], region["camcol"][0], region["field"][0]
print("run:", run, "camcol:", camcol, "field:", field)
bliss_client.load_survey("sdss", run, camcol, field, download_dir=Path("data/sdss"))

##### Get predictions for the downloaded dataset

In [ ]:
est_cat_dl, est_cat_table_dl, pred_tables_dl = bliss_client.predict_sdss(
    data_path="data/sdss",
    weight_save_path="tutorial_encoder/0.pt",
    predict={"dataset": {"run": 1011, "camcol": 3, "fields": [44]}}
)

In [ ]:
bliss_client.plot_predictions_in_notebook()

##### Inspect probabilistic predictions

In [ ]:
print("Number of entries:", len(pred_tables_dl[(1011, 3, 44)]))
pred_tables_dl[(1011, 3, 44)][:5].show_in_notebook(display_length=5)

#### Using sample DECaLS dataset

In [ ]:
est_cat, est_cat_table, pred_tables = bliss_client.predict_decals(
    weight_save_path="tutorial_encoder/single_band_base.pt",
    predict={
        "dataset": {
            "sky_coords": [
                # brick '3366m010' corresponds to SDSS RCF 94-1-12
                {"ra": 336.6643042496718, "dec": -0.9316385797930247},
                # brick '1358p297' corresponds to SDSS RCF 3635-1-169
                {"ra": 135.95496736941683, "dec": 29.646883837721347},
            ]
        }
    },
)

In [ ]:
bliss_client.plot_predictions_in_notebook()